###Day 4:
- Identify and handle all missing values (drop or fill with justification)
- Fix any wrong data types (e.g. numeric columns stored as strings)
- Remove duplicates
- Create 2 new derived columns using .apply() or vectorized operations
- Reshape the data using either .pivot_table() or .groupby() + .agg() to produce a summary table
- Merge it with a second small manually created DataFrame to add extra context (e.g. mapping a column to a category label)

In [128]:
import pandas as pd

df = pd.read_csv('Titanic-Dataset.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


##Identify and handle all missing values drop or fill.

In [129]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


Cabin has a large number of missing values and doesn't have much importance in data so we can drop it.

In [130]:
df= df.drop(columns=['Cabin'])

Age is filled using median becuase it is a numerical feature.



Embarked has only a few missing values, that can be filled using mode.

In [132]:
df['Age'] = df['Age'].fillna(df['Age'].median())

df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [133]:

df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


##Fix any wrong data types (e.g. numeric columns stored as strings)

In [134]:
df.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


We will change data type of age, as it has numerical values, but it's data type is shown as an object.

In [135]:
df['Age'] = df['Age'].astype(int)
df[['Age']].head()

,Age
0,22
1,38
2,26
3,35
4,35


In [136]:
df.Age.dtype

dtype('int64')

##Remove duplicates

In [137]:
df.duplicated().sum()

np.int64(0)

No Duplicates were found. However the process of removing duplicates is mentioned below.

In [138]:
df = df.drop_duplicates()
print(f'Rows after removing duplicates: {len(df)}')

Rows after removing duplicates: 891


##Create 2 new derived columns using .apply() or vectorized operations

Categorize passengers according to the age group
Child, Adult and Seniors

In [139]:
def age_group(age):
  if age < 18:
    return 'Child'
  elif age >= 18 and age < 60:
    return 'Adult'
  else:
    return 'Senior'

# Create the missing column by applying the function
df['age_group'] = df['Age'].apply(age_group)

display(df[['Age', 'age_group']].head(10))

,Age,age_group
0,22,Adult
1,38,Adult
2,26,Adult
3,35,Adult
4,35,Adult
5,28,Adult
6,54,Adult
7,2,Child
8,27,Adult
9,14,Child


####Categorize according to the family size
family_size = sibsp + parch + 1 (the passenger themself)

fare_per_person = fare / family_size. Since fare in this dataset is often the **total ticket price for a family group**, dividing by family size gives a more accurate per-individual cost, useful for fair comparisons across passengers.

In [140]:
df['family_size'] = df['SibSp'] + df['Parch'] + 1
df['fare_per_person'] = (df['Fare'] / df['family_size']).round(2)

df[['SibSp', 'Parch', 'family_size', 'Fare', 'fare_per_person']].head()

,SibSp,Parch,family_size,Fare,fare_per_person
0,1,0,2,7.2500,3.62
1,1,0,2,71.2833,35.64
2,0,0,1,7.9250,7.92
3,1,0,2,53.1000,26.55
4,0,0,1,8.0500,8.05


## Reshape the data using either .pivot_table() or .groupby() + .agg() to produce a summary table

In [141]:
summary_table = df.pivot_table(values='Survived', index='Pclass', columns='Sex', aggfunc='mean')
print("Survival Rate Summary:")
display(summary_table.map(lambda x: f'{x:.2%}'))

Survival Rate Summary:


Sex,female,male
Pclass,,
1,96.81%,36.89%
2,92.11%,15.74%
3,50.00%,13.54%


## Merge it with a second small manually created DataFrame to add extra context (e.g. mapping a column to a category label)
Creating a mapping for 'Pclass' to provide more descriptive labels.

In [142]:
class_mapping = pd.DataFrame({
    'Pclass': [1, 2, 3],
    'Class_Description': ['Luxury Class', 'Standard Class', 'Economy Class']
})

# To prevent MergeError on re-run
if 'Class_Description' in df.columns:
    df = df.drop(columns=['Class_Description'])

# Merge with the main DataFrame
df = df.merge(class_mapping, on='Pclass', how='left')

display(df[['Name', 'Pclass', 'Class_Description']].head())

,Name,Pclass,Class_Description
0,"Braund, Mr. Owen Harris",3,Economy Class
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,Luxury Class
2,"Heikkinen, Miss. Laina",3,Economy Class
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,Luxury Class
4,"Allen, Mr. William Henry",3,Economy Class
